# PROSIT — C'est quoi le problème ?

## 1. Contexte

Dans la continuité des deux prosits précédents, on ne cherche plus seulement à modéliser le problème de tournée, mais à proposer une méthode de résolution exploitable. Le contexte reste celui des tournées de livraison étudiées comme un **TSP métrique** : il faut visiter chaque ville en minimisant le coût total, tout en sachant désormais que ce problème est difficile à résoudre exactement à grande échelle. Agathe recommande donc d’étudier d’abord une modélisation en Programmation Linéaire en Nombres Entiers, puis d’analyser les limites de cette approche ainsi que celles de la programmation dynamique, avant d’envisager d’autres méthodes.


## 2. Mots inconnus

**Programmation linéaire** : méthode d’optimisation mathématique qui consiste à minimiser ou maximiser une fonction objectif sous des contraintes linéaires.

**PLNE** : Programmation Linéaire en Nombres Entiers ; méthode consistant à modéliser un problème avec une fonction objectif, des contraintes linéaires et des variables entières.

**Fonction objectif** : expression mathématique que l’on cherche à minimiser ou maximiser, par exemple un coût, un temps ou une distance.

**Solveur** : logiciel capable de résoudre automatiquement un modèle mathématique, par exemple CBC, GLPK, Gurobi ou CPLEX.

**Simplexe** : algorithme classique utilisé pour résoudre des problèmes de programmation linéaire continue.

**Programmation dynamique** : approche algorithmique qui résout un problème en le découpant en sous-problèmes mémorisés pour éviter des recalculs inutiles.

**Held-Karp** : algorithme exact de programmation dynamique pour le TSP, plus efficace que la force brute mais restant exponentiel.

**Heuristique** : méthode pratique qui cherche rapidement une bonne solution sans garantir qu’elle soit optimale.

**Méta-heuristique** : stratégie générale d’exploration qui guide des heuristiques pour trouver de bonnes solutions à des problèmes complexes.

**Solution sous-optimale** : solution non garantie optimale mais suffisamment bonne et obtenue dans un temps acceptable.


## 3. Problématique

Quelle stratégie de résolution faut-il retenir pour le problème de tournée de livraison modélisé comme un TSP métrique, sachant qu'il est NP-complet ?


## 4. Plan d’action

### 4.1 Repartir du TSP métrique déjà identifié - 1 h 30

Rappeler le cadre du TSP métrique déjà étudié : villes, distances, contraintes de tournée et critère d’optimisation, afin de comparer ensuite les méthodes de résolution.

### 4.2 Formuler le TSP en PLNE - 2 h

Traduire le problème de tournée en fonction objectif et en contraintes linéaires, puis préparer un format compatible avec un solveur.

### 4.3 Résoudre une petite instance par programmation dynamique - 2 h

Présenter le principe de la programmation dynamique, notamment via Held-Karp pour le TSP, puis l’appliquer sur une petite instance afin de comparer cette logique avec la PLNE.

### 4.4 Comparer les limites des méthodes exactes - 1 h 30

Comparer les deux approches selon la taille des instances, la consommation mémoire, le temps de calcul et leur capacité à traiter des tournées avec de nombreuses villes.

### 4.5 Retenir la stratégie adaptée au projet - 1 h

Conclure sur le choix retenu pour le TSP : utiliser une approche heuristique comme méthode principale de production, et réserver la PLNE ou la programmation dynamique aux petites instances et à la validation.


## 5. Réalisation

### 5.1 Repartir du TSP métrique déjà identifié

Dans la continuité du PROSIT 2, le problème métier correspond à une **tournée de livraison** modélisée comme un **TSP métrique** : il faut visiter chaque ville une seule fois et revenir au point de départ en minimisant le coût total. L’objectif ici n’est donc plus de redéfinir tout le problème, mais de choisir une méthode de résolution adaptée à ce cadre.

Pour comparer les approches de résolution, il faut rappeler les éléments suivants :

- ensemble des villes et matrice des distances ;
- variables de décision ;
- contraintes de visite et de tournée ;
- critère d’optimisation ;
- taille attendue des instances.

Exemple de formalisation simplifiée pour le TSP :

- ensemble $V$ : villes à visiter ;
- distance $d_{ij}$ : coût de l’arête entre les villes $i$ et $j$ ;
- variable binaire $x_{ij} \in \{0,1\}$ avec $i < j$ : 1 si l’arête entre $i$ et $j$ appartient à la tournée, 0 sinon.

Dans les contraintes ci-dessous, on utilisera aussi la notation $S \subset V$. Elle désigne simplement un groupe de villes pris à l’intérieur de l’ensemble total des villes. Cette notation sert à empêcher la formation de sous-tours, c’est-à-dire de petits cycles séparés qui ne couvriraient qu’une partie des villes au lieu de former une tournée unique.

On cherche alors à minimiser le coût total de la tournée :

$$
\min \sum_{i < j} d_{ij} x_{ij}
$$

sous :

$$
\sum_{j < i} x_{ji} + \sum_{j > i} x_{ij} = 2 \quad \forall i \in V
$$

$$
\sum_{i < j,\ i,j \in S} x_{ij} \leq |S| - 1 \quad \forall S \subset V,\ 2 \leq |S| \leq |V|-2
$$

$$
x_{ij} \in \{0,1\}
$$

La contrainte de degré impose que chaque ville soit reliée à exactement deux arêtes de la tournée. La contrainte sur les sous-ensembles sert à empêcher la formation de plusieurs petits cycles séparés. Cette formalisation montre donc comment le TSP étudié dans les prosits précédents peut être traduit en PLNE.


In [1]:
cities = ["A", "B", "C", "D"]
adjacency_matrix = [
    [0, 10, 25, 40],
    [10, 0, 15, 30],
    [25, 15, 0, 15],
    [40, 30, 15, 0],
]


**Validation attendue**

- vérification que chaque contrainte métier a une traduction mathématique ;
- revue croisée de la formalisation par un autre membre du groupe ;
- jeu de cas simple calculable à la main pour vérifier le sens du modèle.


### 5.2 Formuler le TSP en PLNE avec PuLP

Une fois le problème formalisé, la mise en œuvre la plus concrète consiste à produire un modèle PLNE exécutable avec un solveur réel. Pour une première mise en œuvre, un bon choix est d’utiliser **Python + PuLP** ou **Python + OR-Tools**. PuLP est un bon point de départ car sa syntaxe est lisible et reste indépendante du solveur utilisé.

Dans le problème réel issu du PROSIT 2, on manipule des variables de type $x_{ij}$ pour représenter les **arêtes** retenues dans la tournée. L’exemple ci-dessous reprend donc une **petite instance TSP** sur 4 villes, décrite par une matrice d’adjacence pondérée. Le but n’est pas de coder un solveur nous-mêmes, mais de montrer comment la formulation mathématique du TSP se traduit en code : les données d’entrée deviennent une matrice de distances, les variables de décision deviennent des arêtes binaires, la fonction objectif minimise la distance totale et les contraintes garantissent une vraie tournée.


In [2]:
# !pip install pulp
from itertools import combinations
from pulp import *

# Donnees d'entree : graphe non oriente pondere, represente par une matrice d'adjacence.
n = len(cities)
edges = [(i, j) for i in range(n) for j in range(i + 1, n)]

def distance(i, j):
    return adjacency_matrix[i][j]

# Variables du probleme : une variable binaire par arete du graphe.
x = {
    (i, j): LpVariable(f"X_{i}_{j}", cat="Binary")
    for (i, j) in edges
}

# Probleme : minimiser la distance totale de la tournee.
prob = LpProblem("TSP_non_oriente", LpMinimize)

# Fonction objectif.
prob += lpSum(distance(i, j) * x[(i, j)] for (i, j) in edges)

# Chaque ville doit etre incidente a exactement deux aretes de la tournee.
for i in range(n):
    prob += lpSum(x[(a, b)] for (a, b) in edges if a == i or b == i) == 2

# Contraintes d'elimination des sous-tours :
# pour tout sous-ensemble S, on interdit de former un cycle ferme strictement a l'interieur de S.
for size in range(2, n - 1):
    for subset in combinations(range(n), size):
        subset_edges = [(i, j) for (i, j) in edges if i in subset and j in subset]
        prob += lpSum(x[(i, j)] for (i, j) in subset_edges) <= len(subset) - 1

# Resolution avec CBC.
prob.solve(PULP_CBC_CMD(msg=False))

# Aretes retenues dans la solution.
selected_edges = [(i, j) for (i, j) in edges if x[(i, j)].varValue == 1]

# Reconstruction de la tournee a partir des aretes choisies.
neighbors = {i: [] for i in range(n)}
for i, j in selected_edges:
    neighbors[i].append(j)
    neighbors[j].append(i)

route_indices = [0]
previous = None
current = 0

while True:
    next_candidates = [node for node in neighbors[current] if node != previous]
    next_node = next_candidates[0]
    route_indices.append(next_node)
    if next_node == 0:
        break
    previous, current = current, next_node

route = [cities[i] for i in route_indices]
plne_solution = {"route": route, "cost": int(value(prob.objective))}

# Affichage du resultat.
print(LpStatus[prob.status])
print("Distance optimale =", plne_solution["cost"])
for v in prob.variables():
    if v.varValue == 1:
        print(f"{v.name}={int(v.varValue)}", end=", ")
print()
print("Aretes retenues :", [(cities[i], cities[j]) for (i, j) in selected_edges])
print("Tournee :", " -> ".join(route))


Optimal
Distance optimale = 80
X_0_1=1, X_0_2=1, X_1_3=1, X_2_3=1, 
Aretes retenues : [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D')]
Tournee : A -> B -> D -> C -> A


Comment lire ce code :

- `cities` et `adjacency_matrix` representent le graphe non oriente et sa matrice d’adjacence ;
- `LpProblem(..., LpMinimize)` indique que l’on cherche a minimiser la distance totale ;
- `x[(i, j)]` cree une variable binaire explicite pour chaque arete, nommee `X_i_j` ;
- `lpSum(...)` permet d’ecrire directement la fonction objectif et les contraintes du modele ;
- les contraintes de degre imposent que chaque ville soit reliee a exactement deux aretes retenues ;
- les contraintes sur les sous-ensembles servent a eliminer les sous-tours ;
- `selected_edges` et `route` servent ensuite a reconstruire la tournee obtenue.


Cet exemple reste volontairement petit, mais il correspond bien au probleme du TSP sur graphe non oriente : on choisit des aretes entre villes, on minimise la distance totale et on ajoute une contrainte pour eviter les sous-tours.

Ce choix permet aussi de travailler dans un environnement reproductible et proche d’un contexte de projet réel.



### 5.3 Résoudre une petite instance par programmation dynamique

Pour compléter la PLNE, on peut résoudre la **même petite instance TSP** par programmation dynamique. Dans le cas du TSP, cette logique correspond à l’algorithme de **Held-Karp** : on mémorise les meilleurs coûts obtenus pour des sous-ensembles de villes, puis on construit progressivement la meilleure tournée.


In [3]:
from itertools import combinations

def held_karp_tsp(cities, start_index):
    other_cities = [i for i in range(len(cities)) if i != start_index]
    dp = {}

    # Cas de base : partir de la ville de depart et rejoindre une seule autre ville.
    for city in other_cities:
        dp[(frozenset([city]), city)] = (distance(start_index, city), [start_index, city])

    # Construction progressive des meilleurs chemins pour chaque sous-ensemble.
    for size in range(2, len(other_cities) + 1):
        for subset in combinations(other_cities, size):
            subset_key = frozenset(subset)
            for last in subset:
                previous_subset = subset_key - {last}
                candidates = []

                for previous_city in previous_subset:
                    previous_cost, previous_path = dp[(previous_subset, previous_city)]
                    candidates.append(
                        (previous_cost + distance(previous_city, last), previous_path + [last])
                    )

                dp[(subset_key, last)] = min(candidates, key=lambda item: item[0])

    full_subset = frozenset(other_cities)
    best_cost, best_path = min(
        (dp[(full_subset, last)][0] + distance(last, start_index), dp[(full_subset, last)][1] + [start_index])
        for last in other_cities
    )

    return {"route": [cities[i] for i in best_path], "cost": best_cost}


In [4]:
dp_solution = held_karp_tsp(cities, start_index=0)
print("Distance optimale par programmation dynamique :", dp_solution["cost"])
print("Tournee :", " -> ".join(dp_solution["route"]))


Distance optimale par programmation dynamique : 80
Tournee : A -> B -> D -> C -> A


Pourquoi cette étape est utile :

- elle montre concrètement le principe de la programmation dynamique sur le TSP ;
- elle permet de comparer le resultat de Held-Karp avec celui de la PLNE sur la meme instance ;
- elle met deja en evidence que cette approche depend fortement de la taille de l’espace d’etats.


In [5]:
plne_value = plne_solution["cost"]
print({
    "PLNE": plne_value,
    "programmation_dynamique": dp_solution["cost"]
})


{'PLNE': 80, 'programmation_dynamique': 80}


### 5.4 Comparer les limites des méthodes exactes

Pour la **PLNE**, les limites apparaissent surtout lorsque le nombre de variables, de contraintes et de relations entre villes devient trop important.

Pour la **programmation dynamique**, les limites apparaissent lorsque l’espace d’états devient trop grand ou trop coûteux à stocker. Dans le cas du TSP, c’est précisément ce qui rend Held-Karp exact mais encore exponentiel.

Dans les deux cas, on obtient des approches exactes intéressantes pour comprendre le problème et traiter de petites tournées, mais elles peuvent devenir trop coûteuses dès que le nombre de villes augmente.

En s’appuyant sur l’analyse du PROSIT 2, on peut retenir un ordre de grandeur simple :

- jusqu’à environ **20 villes**, une méthode exacte reste envisageable pour des tests ou des validations ;
- au-delà, le coût de calcul devient rapidement trop élevé pour un usage courant ;
- pour des tournées avec **plusieurs dizaines ou centaines de villes**, il faut privilégier une méthode heuristique.

Ces seuils restent indicatifs : ils dépendent de l’implémentation, du solveur, de la machine utilisée et de la structure de l’instance.


In [6]:
import time

start = time.perf_counter()
model.solve(pulp.PULP_CBC_CMD(msg=False))
plne_value = int(pulp.value(model.objective))
plne_duration = time.perf_counter() - start

start = time.perf_counter()
dp_solution = held_karp_tsp(cities, start_index=0)
dp_duration = time.perf_counter() - start

print({
    "PLNE": {"value": plne_value, "duration_s": round(plne_duration, 6)},
    "programmation_dynamique": {"value": dp_solution["cost"], "duration_s": round(dp_duration, 6)}
})


NameError: name 'model' is not defined

Comparaison réaliste :

- **PLNE** : très bonne pour modéliser les contraintes du TSP, utile surtout sur de petites instances ou pour valider un modèle ;
- **programmation dynamique** : pertinente pour comprendre une méthode exacte comme Held-Karp, mais limitée par l’explosion de l’espace d’états ;
- **heuristiques / méta-heuristiques** : les seules approches réellement compatibles avec une utilisation rapide sur des tournées de plusieurs dizaines ou centaines de villes.


### 5.5 Retenir la stratégie choisie pour le TSP

Le choix retenu pour le projet est une **approche heuristique**. Ce choix est cohérent avec ce qui a été montré dans le PROSIT 2 : le TSP est trop coûteux à résoudre exactement dès que le nombre de villes augmente. La **PLNE** et la **programmation dynamique** restent utiles pour comprendre le problème, construire un modèle rigoureux et valider de petites instances, mais elles ne constituent pas la solution principale à retenir pour une exploitation à grande échelle.

En pratique, on peut formuler la règle de décision ainsi :

- pour **de petites instances** (de l’ordre de **20 villes ou moins**), une méthode exacte peut encore servir à vérifier ou comparer les solutions ;
- pour **des instances réelles** avec **plusieurs dizaines ou centaines de villes**, on retient une **heuristique** comme méthode principale ;
- si la qualité d’une heuristique simple n’est pas suffisante, on pourra la renforcer par une **méta-heuristique** ou une amélioration locale.


In [ ]:
def nearest_neighbor_tsp(cities, start_index):
    unvisited = set(range(len(cities)))
    unvisited.remove(start_index)
    route_indices = [start_index]
    total_cost = 0
    current = start_index

    while unvisited:
        next_city = min(unvisited, key=lambda city: distance(current, city))
        total_cost += distance(current, next_city)
        route_indices.append(next_city)
        unvisited.remove(next_city)
        current = next_city

    total_cost += distance(current, start_index)
    route_indices.append(start_index)

    return {"route": [cities[i] for i in route_indices], "cost": total_cost}

nearest_neighbor_tsp(cities, start_index=0)


**Décision retenue**

- la **stratégie retenue** pour le projet est une **heuristique** de résolution du TSP ;
- la **PLNE** sert de référence pour modéliser rigoureusement le problème et tester des instances d’environ **20 villes ou moins** ;
- la **programmation dynamique** sert surtout à comprendre une autre approche exacte, mais ne constitue pas le choix principal au-delà des petites tailles ;
- une **méta-heuristique** peut être envisagée ensuite si une heuristique simple ne donne pas une qualité suffisante.

En conclusion, le problème étant déjà identifié comme un TSP métrique difficile, la bonne démarche n’est pas de retenir une méthode exacte comme solution principale. La décision la plus cohérente est d’utiliser une **heuristique** pour produire rapidement une tournée exploitable sur des instances réelles, et de réserver la **PLNE** ou la **programmation dynamique** aux petites tailles, typiquement autour de **20 villes ou moins**, pour l’analyse et la validation.
